# Olist: ratio metric, delta method và thí nghiệm A/A

Đội pricing của Olist muốn thử một thay đổi trên trang sản phẩm và đo **giá trung bình mỗi sản phẩm
bán ra** (average item price):

$$
R = \frac{\text{tổng giá các sản phẩm bán ra}}{\text{số sản phẩm bán ra}}
$$

Thí nghiệm random theo **khách** (`customer_unique_id`), nhưng metric tính theo **sản phẩm**. Đơn vị
phân tích nhỏ hơn đơn vị random: các sản phẩm của cùng một khách không độc lập. Notebook này đo
xem bỏ qua điều đó sai bao nhiêu trên dữ liệu thật, bằng thí nghiệm A/A (không có tác động) chạy
lại hàng nghìn lần trên chính dữ liệu Olist.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` đọc dữ liệu thật và chạy mô phỏng. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/04-advanced-experimentation/exercises/intermediate/02-olist-ratio-aa -q`

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chisquare, norm

## 1. Đọc dữ liệu

`load_items(raw_dir) -> DataFrame`: mỗi dòng một sản phẩm bán ra trong **đơn giao xong**
(`order_status == "delivered"` và có `order_delivered_customer_date`), cột `order_id`,
`customer_unique_id`, `price`. Cần ba tệp: orders, customers (để nối `customer_id` sang
`customer_unique_id`), order_items.

In [ ]:
def load_items(raw_dir: Path) -> pd.DataFrame:
    raw_dir = Path(raw_dir)
    orders = pd.read_csv(raw_dir / "olist_orders_dataset.csv")
    customers = pd.read_csv(raw_dir / "olist_customers_dataset.csv")
    items = pd.read_csv(raw_dir / "olist_order_items_dataset.csv")
    done = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    done = done.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    return items.merge(done[["order_id", "customer_unique_id"]], on="order_id")[["order_id", "customer_unique_id", "price"]]

In [ ]:
# explore: tải bằng uv run python data/download_olist.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
items = load_items(ROOT / "data/raw")
per_customer = items.groupby("customer_unique_id").size()
print(len(items), "sản phẩm,", items.order_id.nunique(), "đơn,", len(per_customer), "khách")
print("Số sản phẩm mỗi khách:", per_customer.describe().round(2).to_dict())
print("Tỉ lệ khách có từ 2 sản phẩm:", round((per_customer > 1).mean(), 4))

## 2. Phân nhóm theo khách

`hash_assign(ids, salt, share=0.5) -> np.ndarray[bool]`: giống hàm `assign` ở module 03, băm
`f"{salt}:{id}"` bằng SHA-256, lấy 8 ký tự hex đầu chia cho `0xFFFFFFFF`, nhỏ hơn `share` thì vào
treatment. Cùng khách luôn cùng nhóm, không cần lưu bảng.

In [ ]:
def hash_assign(ids, salt: str, share: float = 0.5) -> np.ndarray:
    def bucket(i) -> float:
        return int(hashlib.sha256(f"{salt}:{i}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return np.array([bucket(i) < share for i in ids], dtype=bool)

## 3. Bảng theo khách và delta method

- `customer_table(items) -> DataFrame`: index `customer_unique_id`, cột `revenue` (tổng `price`) và
  `n_items` (số sản phẩm).
- `delta_test(table, treat) -> dict`: `ratio_a`, `ratio_b`, `diff`, `se`, `p_value` cho
  $$R_B - R_A$$, SE bằng delta method trên bảng theo khách (công thức bài Basic 02). `treat` là
  mảng bool cùng thứ tự với `table`.
- `naive_test(prices_a, prices_b) -> dict`: `diff`, `se`, `p_value` coi **mỗi sản phẩm** là một
  quan sát độc lập (Welch trên mức sản phẩm). Đây là cách tính sai, để so sánh.

In [ ]:
def customer_table(items: pd.DataFrame) -> pd.DataFrame:
    return items.groupby("customer_unique_id").agg(revenue=("price", "sum"), n_items=("price", "size"))


def _ratio_var(num: np.ndarray, den: np.ndarray) -> tuple[float, float]:
    ratio = num.sum() / den.sum()
    c = np.cov(num, den)
    return ratio, (c[0, 0] - 2 * ratio * c[0, 1] + ratio**2 * c[1, 1]) / (len(num) * den.mean() ** 2)


def delta_test(table: pd.DataFrame, treat: np.ndarray) -> dict:
    num = table.revenue.to_numpy(dtype=float)
    den = table.n_items.to_numpy(dtype=float)
    treat = np.asarray(treat, dtype=bool)
    ra, va = _ratio_var(num[~treat], den[~treat])
    rb, vb = _ratio_var(num[treat], den[treat])
    se = float(np.sqrt(va + vb))
    return {"ratio_a": float(ra), "ratio_b": float(rb), "diff": float(rb - ra), "se": se,
            "p_value": float(2 * norm.sf(abs(rb - ra) / se))}


def naive_test(prices_a: np.ndarray, prices_b: np.ndarray) -> dict:
    diff = float(np.mean(prices_b) - np.mean(prices_a))
    se = float(np.sqrt(np.var(prices_a, ddof=1) / len(prices_a) + np.var(prices_b, ddof=1) / len(prices_b)))
    return {"diff": diff, "se": se, "p_value": float(2 * norm.sf(abs(diff) / se))}

In [ ]:
# explore: một lần phân nhóm thật bằng hash, kiểm SRM, so hai cách tính SE
table = customer_table(items)
treat = hash_assign(table.index.to_numpy(), "olist-aov-aa")
print("Khách mỗi nhánh:", int((~treat).sum()), int(treat.sum()), "SRM p =", round(chisquare([(~treat).sum(), treat.sum()]).pvalue, 4))
out = delta_test(table, treat)
in_b = items.customer_unique_id.isin(table.index[treat])
naive = naive_test(items.price[~in_b].to_numpy(), items.price[in_b].to_numpy())
print({k: round(v, 4) for k, v in out.items()})
print({k: round(v, 4) for k, v in naive.items()})
print("design effect (var delta / var naive):", round((out["se"] / naive["se"]) ** 2, 3))

## 4. Thí nghiệm A/A lặp lại

`aa_study(items, n_sims=200, seed=0) -> dict` với hai khoá `delta` và `naive`: tỉ lệ số lần
p < 0.05 qua `n_sims` lần chia ngẫu nhiên **khách** 50/50 (dùng `np.random.default_rng(seed)`,
mỗi lần `rng.random(số khách) < 0.5`). Không có tác động nào, nên một phương pháp đúng phải cho
khoảng 5%.

Cần chạy đủ nhanh: 2 000 lần trên 93 nghìn khách trong vài chục giây. Tính sẵn bảng theo khách và
mã số khách của từng sản phẩm một lần, mỗi vòng chỉ làm phép toán numpy.

In [ ]:
def aa_study(items: pd.DataFrame, n_sims: int = 200, seed: int = 0) -> dict:
    table = customer_table(items)
    code_of_item = pd.Categorical(items.customer_unique_id, categories=table.index).codes
    prices = items.price.to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    hits = {"delta": 0, "naive": 0}
    for _ in range(n_sims):
        treat = rng.random(len(table)) < 0.5
        hits["delta"] += delta_test(table, treat)["p_value"] < 0.05
        item_treat = treat[code_of_item]
        hits["naive"] += naive_test(prices[~item_treat], prices[item_treat])["p_value"] < 0.05
    return {k: v / n_sims for k, v in hits.items()}

In [ ]:
# explore: 2 000 lần A/A trên Olist thật (khoảng 10 giây)
import time

start = time.time()
rates = aa_study(items, n_sims=2000, seed=0)
print(rates, f"{time.time() - start:.1f}s")
print("Kỳ vọng từ design effect:", round(2 * norm.sf(1.96 / np.sqrt((out['se'] / naive['se']) ** 2)), 3))

## 5. Cài thêm tác động thật

Nhân `revenue` của khách nhóm treatment với 1.02 (giá tăng 2%), lặp lại 500 lần: delta method phát
hiện được bao nhiêu phần trăm? Đó là power thật của thí nghiệm này với toàn bộ dữ liệu Olist.

In [ ]:
# explore: power với tác động +2% cài vào dữ liệu thật
rng = np.random.default_rng(1)
detected = 0
for _ in range(500):
    t = rng.random(len(table)) < 0.5
    lifted = table.assign(revenue=np.where(t, table.revenue * 1.02, table.revenue))
    detected += delta_test(lifted, t)["p_value"] < 0.05
print("power ở +2%:", detected / 500)

## 6. Kết luận (bài mẫu)

Trên Olist, coi mỗi sản phẩm là một quan sát độc lập làm variance bị đánh giá thấp khoảng 17%
(design effect khoảng 1.17), nên A/A báo "có ý nghĩa" khoảng 7% số lần thay vì 5%; delta method
giữ đúng quanh 5%. Sai lệch nhỏ vì 97% khách chỉ mua một lần và phần lớn đơn chỉ có một sản phẩm:
cụm rất nhỏ. Trên một sản phẩm mà mỗi user có hàng chục phiên hoặc lượt xem (app, trang tin tức),
design effect lớn hơn nhiều và cách tính ngây thơ có thể cho tỉ lệ báo động nhầm vài chục phần
trăm (bài Basic 02 mô phỏng ra khoảng 25%).

Chọn OEC: "giá trung bình mỗi sản phẩm" dễ bị hiểu sai (giá tăng nhưng khách mua ít đi thì doanh
thu giảm), nên nó hợp làm metric chẩn đoán. OEC nên là doanh thu (hoặc lợi nhuận) **mỗi khách được
random**, metric có đơn vị trùng đơn vị random, không cần delta method và trả lời đúng câu hỏi kinh
doanh. Delta method vẫn cần cho mọi metric chẩn đoán dạng tỉ số đi kèm.